In [1]:
# --- Étape 1 : Installation des bibliothèques ---
!pip install yfinance pandas numpy matplotlib seaborn

# --- Étape 2 : Import des librairies ---
import yfinance as yf
import pandas as pd
import numpy as np
from google.colab import files

# --- Étape 3 : Définir les cryptos et la période ---
cryptos = ['BTC-USD','ETH-USD','BNB-USD','SOL-USD','ADA-USD','DOGE-USD','XRP-USD']
start_date = '2021-01-01'
end_date = '2025-10-30'

# --- Étape 4 : Télécharger les données ---
raw = yf.download(cryptos, start=start_date, end=end_date, auto_adjust=True)

# --- Étape 5 : Fonction RSI ---
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# --- Étape 6 : Calculer les indicateurs pour chaque crypto ---
all_data = []

for crypto in cryptos:
    df = pd.DataFrame()
    df['Date'] = raw['Close'][crypto].index
    df['Crypto'] = crypto
    df['Ticker'] = crypto
    df['Open'] = raw['Open'][crypto].values
    df['High'] = raw['High'][crypto].values
    df['Low'] = raw['Low'][crypto].values
    df['Close'] = raw['Close'][crypto].values
    df['Volume'] = raw['Volume'][crypto].values

    close = pd.Series(raw['Close'][crypto].values)

    # Indicateurs
    df['Return_Daily'] = close.pct_change()
    df['Cumulative_Return'] = (1 + close.pct_change()).cumprod() - 1
    df['Daily_Change'] = close.diff()
    df['Daily_Change_Pct'] = close.pct_change() * 100
    df['MA_20'] = close.rolling(window=20).mean()
    df['MA_50'] = close.rolling(window=50).mean()
    df['MA_200'] = close.rolling(window=200).mean()
    df['Rolling_Volatility_30d'] = close.pct_change().rolling(window=30).std()
    df['RSI_14'] = compute_rsi(close, window=14)
    df['Max_Drawdown'] = (close - close.cummax()) / close.cummax()
    df['High_Low_Range'] = df['High'] - df['Low']
    df['Line_30'] = close.rolling(window=30).mean()

    all_data.append(df)

# --- Étape 7 : Combiner toutes les cryptos ---
final_df = pd.concat(all_data, ignore_index=True)

# --- Étape 8 : Sauvegarder et télécharger ---
final_df.to_csv('cryptos_yahoo_enriched_2021_2025.csv', index=False)
files.download('cryptos_yahoo_enriched_2021_2025.csv')



[*********************100%***********************]  7 of 7 completed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>